# 02 · Leakage-safe feature engineering
Features are computed from transactions before each cutoff. The six-month target is computed from transactions after that cutoff.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from src.data_prep import load_raw, clean_transactions, OUT
from src.features import build_customer_table, FEATURES
tx,audit=clean_transactions(load_raw())
returns=pd.read_csv(OUT/'transactions_returns.csv')
cutoff=pd.Timestamp('2011-03-01')
features=build_customer_table(tx,cutoff,returns=returns)
features.head(), features[FEATURES].describe().T

## Why leakage is prevented
The function filters `InvoiceDate < cutoff` before calculating RFM, tenure, order intervals, distinct products, return rate or Q4 share. It separately filters the target to `[cutoff, cutoff + 6 months)`. Changing any post-cutoff revenue cannot alter the feature columns. A 90-day inactivity label is derived from the target window only and is never used as a feature.

In [ ]:
features[['Customer ID','recency_days','frequency','monetary','future_revenue','inactive_90d']].head(10)